# Practical 2A — Iris MLP with SIGMOID

## Aim
Train an Iris MLP using SIGMOID activation, inspect training loss and validation accuracy, and plot the activation curve.

## Setup
Run the installation cell if required. These notebooks use Python 3.10–3.12 where possible. On Apple Silicon, TensorFlow/PyTorch acceleration depends on the installed build; CPU execution is supported.

**Important:** dataset downloads and model training may take time. Results are generated when you run the notebook; no accuracy values are pre-filled.

In [ ]:
%pip install -q numpy pandas matplotlib scikit-learn tensorflow

## Reproducibility\nA fixed random seed is used where practical. If a dataset download fails, check your internet connection and rerun the loading cell.

## 1. Load and normalize the Iris dataset

In [ ]:
import time, numpy as np, pandas as pd, matplotlib.pyplot as plt, tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score
np.random.seed(42); tf.random.set_seed(42)
iris=load_iris()
X_train,X_test,y_train,y_test=train_test_split(iris.data,iris.target,test_size=0.2,stratify=iris.target,random_state=42)
scaler=StandardScaler()
X_train=scaler.fit_transform(X_train).astype("float32")
X_test=scaler.transform(X_test).astype("float32")
X_tr,X_val,y_tr,y_val=train_test_split(X_train,y_train,test_size=0.2,stratify=y_train,random_state=42)
print(X_tr.shape, X_val.shape, X_test.shape)

## 2. MLP with SIGMOID activation\nThe hidden layer uses the requested activation. Forward propagation is expressed explicitly as Dense → activation → softmax output. Keras automatic differentiation performs backpropagation.

In [ ]:
def activation_fn(x):
    return tf.nn.sigmoid(x)
inputs=keras.Input(shape=(4,))
hidden=layers.Dense(16, activation=activation_fn)(inputs)
outputs=layers.Dense(3, activation="softmax")(hidden)
model=keras.Model(inputs,outputs)
model.compile(optimizer=keras.optimizers.Adam(learning_rate=0.01),
              loss="sparse_categorical_crossentropy",metrics=["accuracy"])
start=time.time()
history=model.fit(X_tr,y_tr,validation_data=(X_val,y_val),epochs=100,batch_size=16,verbose=0)
elapsed=time.time()-start
test_loss,test_acc=model.evaluate(X_test,y_test,verbose=0)
print("Validation accuracy:",max(history.history["val_accuracy"]))
print("Test accuracy:",test_acc,"Test loss:",test_loss,"Training time:",round(elapsed,2),"s")

## 3. Plot training loss and validation accuracy

In [ ]:
plt.figure(figsize=(10,4))
plt.subplot(1,2,1); plt.plot(history.history["loss"]); plt.plot(history.history["val_loss"]); plt.title("Loss"); plt.legend(["Train","Validation"])
plt.subplot(1,2,2); plt.plot(history.history["val_accuracy"]); plt.title("Validation accuracy")
plt.tight_layout(); plt.show()

## 4. Plot the SIGMOID activation curve

In [ ]:
z=np.linspace(-6,6,300)
y=1/(1+np.exp(-z))
plt.plot(z,y); plt.axhline(0,linewidth=0.5); plt.axvline(0,linewidth=0.5)
plt.title("SIGMOID activation"); plt.xlabel("Input"); plt.ylabel("Output"); plt.grid(True); plt.show()

## 5. Manual forward-propagation demonstration

In [ ]:
# Extract learned weights and manually calculate one sample's forward pass.
W1,b1=model.layers[1].get_weights()
W2,b2=model.layers[2].get_weights()
sample=X_test[:1]
z1=sample @ W1 + b1
a1=activation_fn(tf.convert_to_tensor(z1)).numpy()
z2=a1 @ W2 + b2
exp=np.exp(z2-z2.max(axis=1,keepdims=True)); manual_prob=exp/exp.sum(axis=1,keepdims=True)
print("Manual probabilities:",manual_prob)
print("Keras probabilities:",model.predict(sample,verbose=0))

## 6. Conclusion\nSummarize measured validation accuracy, convergence behavior and the shape of the sigmoid curve. Compare with the other activation-function practicals; results may vary with initialization and training.